# PyNvVideoCodec smoke test

PyNvVideoCodec is NVIDIA's Python API for the GPU's video decoder (NVDEC) and encoder (NVENC). The module is named `PyNvVideoCodec`; the distribution is `pynvvideocodec`. Even `import PyNvVideoCodec` needs the NVIDIA driver, so the first code cell is tagged `requires-cuda-driver` and the others `requires-gpu`. The clips are made in the cells by the GPU's own encoder. No network.

## Example 1 — the installed version, the encoder-API module the driver selects, and both variants load

In [ ]:
import importlib.metadata
import os
import subprocess
import sys
import PyNvVideoCodec as nvc

assert importlib.metadata.version("pynvvideocodec") == nvc.__version__, (importlib.metadata.version("pynvvideocodec"), nvc.__version__)
folder = os.path.dirname(nvc.__file__)
# __init__ asks the driver for its NVENC API level and loads the matching one of the two installed modules.
expected = "_130" if nvc.supportedNvEncVersion >= nvc.NVENC_VER_13 else "_121"
assert nvc.module_suffix == expected, (nvc.module_suffix, nvc.supportedNvEncVersion)
assert os.path.basename(nvc._PyNvVideoCodec.__file__).startswith("PyNvVideoCodec" + expected + ".")
# Only one variant is ever loaded in a process, so each is also loaded in its own fresh interpreter, as __init__ does.
load = ("import importlib.util, sys\n"
        "spec = importlib.util.spec_from_file_location('_PyNvVideoCodec', sys.argv[1])\n"
        "module = importlib.util.module_from_spec(spec); spec.loader.exec_module(module)\n"
        "print(len([name for name in dir(module) if not name.startswith('_')]))\n")
variants = sorted(name for name in os.listdir(folder) if name.startswith("PyNvVideoCodec_"))
assert len(variants) == 2, variants
for name in variants:
    done = subprocess.run([sys.executable, "-c", load, os.path.join(folder, name)], capture_output=True, text=True, timeout=120)
    assert done.returncode == 0 and int(done.stdout) > 40, (name, done.stderr[-300:])
print("PyNvVideoCodec", nvc.__version__, "NVENC API level", nvc.supportedNvEncVersion, "-> module", expected, "| both variants load")

## Example 2 — what the GPU's video hardware reports, and an encoder session's stream header

In [ ]:
import PyNvVideoCodec as nvc

decode = nvc.GetDecoderCaps(gpuid=0, codec=nvc.cudaVideoCodec.H264)
assert decode["supported"] == 1 and decode["width_max"] >= 1920 and decode["height_max"] >= 1080, decode
encode = nvc.GetEncoderCaps(gpuid=0, codec="h264")
assert encode["width_max"] >= 1920 and encode["height_max"] >= 1080 and encode["num_encoder_engines"] >= 1, encode
encoder = nvc.CreateEncoder(320, 240, "NV12", True, codec="h264", gop=15, bf=0, fps=30)
header = encoder.GetSequenceParams()
# An H.264 stream header: start code, then a sequence parameter set (NAL type 7), later a picture parameter set (8).
assert header[:4] == b"\x00\x00\x00\x01" and header[4] & 0x1F == 7, header[:8].hex()
assert any(header[i:i + 4] == b"\x00\x00\x00\x01" and header[i + 4] & 0x1F == 8 for i in range(4, len(header) - 4)), header.hex()
assert encoder.GetNumBFrames() == 0
others = {name: nvc.GetDecoderCaps(gpuid=0, codec=getattr(nvc.cudaVideoCodec, name))["supported"] for name in ("HEVC", "AV1", "VP9")}
print(f"H.264 decode up to {decode['width_max']}x{decode['height_max']}, encode up to {encode['width_max']}x{encode['height_max']}; "
      f"stream header {len(header)} bytes (SPS + PPS); other decoders supported: {others}")

## Example 3 — round trip: encode 30 frames with NVENC, read the stream back with FFmpeg, decode with NVDEC

In [ ]:
import pathlib
import tempfile
import numpy as np
import PyNvVideoCodec as nvc

W, H, N = 320, 240, 30
def source_frame(i):                                    # NV12: a moving luma gradient over grey chroma
    luma = ((np.arange(H)[:, None] + np.arange(W)[None, :] + 4 * i) % 256).astype(np.uint8)
    return np.concatenate([luma, np.full((H // 2, W), 128, np.uint8)])
encoder = nvc.CreateEncoder(W, H, "NV12", True, codec="h264", gop=15, bf=0, fps=30)
packets = [p for i in range(N) for p in encoder.Encode(source_frame(i))] + list(encoder.EndEncode())
assert len(packets) == N and packets[0]["picture_type"] == packets[15]["picture_type"] != packets[1]["picture_type"]
assert [p["timestamp"] for p in packets] == list(range(N))
with tempfile.TemporaryDirectory() as folder:
    path = pathlib.Path(folder, "clip.h264")
    path.write_bytes(b"".join(p["data"] for p in packets))
    demuxer = nvc.CreateDemuxer(filename=str(path))
    assert (demuxer.Width(), demuxer.Height(), demuxer.GetNvCodecId()) == (W, H, nvc.cudaVideoCodec.H264)
    decoder = nvc.CreateDecoder(gpuid=0, codec=demuxer.GetNvCodecId(), usedevicememory=False)
    # Copy each frame as it arrives: the decoder reuses its output buffers on the next call.
    frames = [np.from_dlpack(frame).copy() for packet in demuxer for frame in decoder.Decode(packet)]
assert len(frames) == N and frames[0].shape == (H * 3 // 2, W) and frames[0].dtype == np.uint8
errors = [np.abs(frames[i][:H].astype(np.int16) - source_frame(i)[:H].astype(np.int16)).mean() for i in range(N)]
assert max(errors) < 2.0, errors                                   # lossy codec: close, not equal; and in the right order
assert all(int(frame[H:].min()) >= 126 and int(frame[H:].max()) <= 130 for frame in frames)
print(f"{sum(len(p['data']) for p in packets)} bytes of H.264 -> {len(frames)} NV12 frames; largest mean luma error {max(errors):.2f} of 255")

## Example 4 — decoded frames as RGB and planar RGB, checked against the video-range formula

In [ ]:
import pathlib
import tempfile
import numpy as np
import PyNvVideoCodec as nvc

W, H, N = 320, 240, 30
def source_frame(i):                                    # NV12: a moving luma gradient over grey chroma
    luma = ((np.arange(H)[:, None] + np.arange(W)[None, :] + 4 * i) % 256).astype(np.uint8)
    return np.concatenate([luma, np.full((H // 2, W), 128, np.uint8)])
encoder = nvc.CreateEncoder(W, H, "NV12", True, codec="h264", gop=15, bf=0, fps=30)
packets = [p for i in range(N) for p in encoder.Encode(source_frame(i))] + list(encoder.EndEncode())
def decode(path, colour):
    demuxer = nvc.CreateDemuxer(filename=path)
    decoder = nvc.CreateDecoder(gpuid=0, codec=demuxer.GetNvCodecId(), usedevicememory=False, outputColorType=colour)
    return [np.from_dlpack(frame).copy() for packet in demuxer for frame in decoder.Decode(packet)]
with tempfile.TemporaryDirectory() as folder:
    path = str(pathlib.Path(folder, "clip.h264"))
    pathlib.Path(path).write_bytes(b"".join(p["data"] for p in packets))
    native = decode(path, nvc.OutputColorType.NATIVE)
    interleaved = decode(path, nvc.OutputColorType.RGB)
    planar = decode(path, nvc.OutputColorType.RGBP)
assert interleaved[3].shape == (H, W, 3) and planar[3].shape == (3, H, W) and len(interleaved) == len(planar) == N
assert all(np.array_equal(interleaved[i].transpose(2, 0, 1), planar[i]) for i in range(N))      # same pixels, two layouts
# Grey chroma means red = green = blue, and video-range luma (16..235) is stretched to 0..255.
expected = np.clip(np.round(1.164383 * (native[3][:H].astype(np.float64) - 16.0)), 0, 255)
assert np.abs(interleaved[3].astype(np.float64) - expected[:, :, None]).max() <= 1.0
print("RGB (HWC) and planar RGB (CHW) agree for all", N, "frames; every pixel is within 1 of the video-range formula")

## Example 5 — vLLM's call: a clip in an MP4 container, frames picked by index straight into GPU memory

In [ ]:
import importlib.util
import pathlib
import tempfile
import numpy as np
import PyNvVideoCodec as nvc

W, H, N = 320, 240, 30
def source_frame(i):                                    # NV12: a moving luma gradient over grey chroma
    luma = ((np.arange(H)[:, None] + np.arange(W)[None, :] + 4 * i) % 256).astype(np.uint8)
    return np.concatenate([luma, np.full((H // 2, W), 128, np.uint8)])
encoder = nvc.CreateEncoder(W, H, "NV12", True, codec="h264", gop=15, bf=0, fps=30)
packets = [p for i in range(N) for p in encoder.Encode(source_frame(i))] + list(encoder.EndEncode())
with tempfile.TemporaryDirectory() as folder:
    path = str(pathlib.Path(folder, "clip.mp4"))
    muxer = nvc.FFmpegMuxer(path, nvc.MEDIA_FORMAT.MP4, "h264", W, H, 30, 1, 1, 90000, encoder.GetSequenceParams())
    muxer.SetUniformPtsIncrement(3000)                             # 90000 ticks per second / 30 frames per second
    for packet in packets:
        muxer.MuxVideoPacket(packet["data"], packet["picture_type"], packet["timestamp"])
    muxer.Finalize(); del muxer
    # The arguments vLLM passes (vllm/multimodal/video_decoders/pynvvideocodec.py), minus its own CUDA stream.
    decoder = nvc.SimpleDecoder(path, output_color_type=nvc.OutputColorType.RGB, use_device_memory=True, need_scanned_stream_metadata=True, gpu_id=0, decoder_cache_size=1)
    assert (len(decoder), (meta := decoder.get_stream_metadata()).width, meta.height, round(meta.average_fps)) == (N, W, H, 30), (len(decoder), meta.average_fps)
    picked = decoder.get_batch_frames_by_index([29, 0, 10])        # out of order on purpose: returned in the order asked
    # DLPack device type 2 is CUDA: the frames stay in GPU memory. Their time stamps prove which frames came back.
    assert [frame.__dlpack_device__()[0] for frame in picked] == [2, 2, 2] and [frame.getPTS() for frame in picked] == [29 * 3000, 0, 10 * 3000], [frame.getPTS() for frame in picked]
    # torch is vLLM's consumer of these frames, not a dependency of this package, so it is used only where installed.
    torch = importlib.import_module("torch") if importlib.util.find_spec("torch") is not None else None
    assert torch is None or (torch.from_dlpack(picked[0]).shape == (H, W, 3) and torch.from_dlpack(picked[0]).is_cuda)
    print(f"MP4: {N} frames at {meta.average_fps:.0f} fps, codec {meta.codec_name}; frames 29, 0, 10 picked by index into GPU memory" + ("" if torch is None else " and handed to torch without a copy"))